In [9]:
import shutil

In [1]:
import xarray as xr
import json
import numpy as np
from sklearn.decomposition import PCA

In [2]:
settings_file_path="dpa_tune_settings.json"
with open(settings_file_path, 'r') as file:
        settings = json.load(file)

In [3]:
ds = xr.open_dataset(settings['dataset_trefht'])
ds_train = ds.isel(time=slice(0, 128000))
ds_train

<xarray.Dataset> Size: 525MB
Dimensions:  (lat: 32, lon: 32, time: 128000)
Coordinates:
  * lon      (lon) float64 256B -11.25 -10.0 -8.75 -7.5 ... 25.0 26.25 27.5
  * lat      (lat) float64 256B 34.4 35.34 36.28 37.23 ... 61.73 62.67 63.61
  * time     (time) object 1MB 1850-06-02 00:00:00 ... 2060-08-16 00:00:00
Data variables:
    TREFHT   (lat, lon, time) float32 524MB ...

In [4]:
import pca_encoder as pcae

pca, pcs = pcae.get_PC(ds_train)

(128000, 1024)
Shape after removing NANs: (128000, 648)


In [8]:
pca.components_

array([[ 1.0301703e-02,  2.3001723e-02,  2.6002122e-02, ...,
         3.5252731e-02,  3.5435781e-02,  3.5778224e-02],
       [ 3.9741132e-02,  5.2832507e-02,  5.0956808e-02, ...,
        -6.5520853e-02, -6.6108301e-02, -6.6306911e-02],
       [-1.7425623e-02, -8.3462549e-03, -9.5677597e-04, ...,
        -1.9813394e-02, -1.7822031e-02, -1.5945619e-02],
       ...,
       [ 8.5182646e-06,  3.7391589e-04, -1.2399087e-03, ...,
         3.9130729e-03, -1.9435765e-03,  1.1101663e-03],
       [ 1.6132179e-04, -3.9315195e-04,  6.8122562e-04, ...,
        -5.6348386e-04,  1.4817523e-03, -8.9327520e-04],
       [ 3.1371828e-04,  1.8238634e-04,  3.9662096e-05, ...,
        -1.6832174e-03,  2.1088498e-03, -1.0672620e-04]],
      shape=(648, 648), dtype=float32)

In [6]:
eofs = xr.DataArray(
    data=pca.components_,
    dims=["mode", "space"],
    coords={"mode": np.arange(pcs.shape[1])}
)#.unstack("space")  # reshape back to lat/lon

pcs_xr = xr.DataArray(
    data=pcs,
    dims=["time", "mode"],
    coords={"time": ds_train["time"], "mode": np.arange(pcs.shape[1])}
)


In [ ]:
eofs